# Bank Marketing — EDA + model bake-off (Slice 1)

Drives the platform-track training narrative. All reusable logic lives in `ml/data.py`, `ml/features.py`, `ml/train.py`, `ml/evaluate.py`. This notebook is allowed to be exploratory and untracked-by-CI; the CLI `python -m ml.train` is the source of truth.

Run from the `platform/` directory.

In [ ]:
from pathlib import Path

import pandas as pd

from ml.data import (
    clean,
    load_raw,
    split,
    split_xy,
    validate_columns,
)
from ml.evaluate import evaluate_at_threshold, tune_threshold
from ml.train import cv_compare, fit_final, majority_baseline_scores

DATA_PATH = Path("data/raw/bank-additional-full.csv")
OUT_DIR = Path("data/processed/")

## 1. Load + validate

In [ ]:
raw = load_raw(DATA_PATH)
validate_columns(raw)
print(f"rows={len(raw):,}  cols={raw.shape[1]}")
raw.head()

## 2. Quick EDA on the raw frame

Target balance, `'unknown'` rate per categorical, and `pdays == 999` rate.

In [ ]:
print("positive rate (y==yes):", (raw["y"] == "yes").mean())
print("pdays == 999 rate:     ", (raw["pdays"] == 999).mean())

In [ ]:
unknown_rates = (
    raw.select_dtypes(include="object")
    .apply(lambda s: (s == "unknown").mean())
    .sort_values(ascending=False)
)
unknown_rates[unknown_rates > 0]

## 3. Clean + stratified 60/20/20 split

Encodes the data-prep rules locked by `tests/test_data.py`.

In [ ]:
cleaned = clean(raw)
train_df, val_df, test_df = split(cleaned, random_state=42)
X_train, y_train = split_xy(train_df)
X_val, y_val = split_xy(val_df)
X_test, y_test = split_xy(test_df)

pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "rows": [len(train_df), len(val_df), len(test_df)],
        "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    }
)

## 4. Majority-class baseline

Sanity floor — anything we ship must beat this on `f1_macro` and `roc_auc`.

In [ ]:
baseline = majority_baseline_scores(X_train, y_train)
baseline

## 5. 5-fold CV bake-off

Logistic regression, random forest, gradient boosting. Mean ± std for accuracy, macro F1, ROC-AUC.

In [ ]:
cv_table = cv_compare(X_train, y_train)
cv_table

## 6. Fit final pipeline + tune threshold on validation

Threshold rule: highest threshold whose recall on val >= 0.75.

In [ ]:
chosen = cv_table.iloc[0]["model"]
print("chosen model:", chosen)

pipe = fit_final(chosen, X_train, y_train)
val_proba = pipe.predict_proba(X_val)[:, 1]
threshold = tune_threshold(y_val.to_numpy(), val_proba, min_recall=0.75)
print(f"tuned threshold: {threshold:.4f}")
evaluate_at_threshold(y_val.to_numpy(), val_proba, threshold)

## 7. Single test-set evaluation

Don't peek at this until the threshold is locked.

In [ ]:
test_proba = pipe.predict_proba(X_test)[:, 1]
evaluate_at_threshold(y_test.to_numpy(), test_proba, threshold)

## 8. Persist artifacts (Slice 1 handoff)

Local `model.joblib` is the input to Slice 2 (MLflow registration). Both the joblib file and processed parquets are gitignored.

In [ ]:
import joblib

OUT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_parquet(OUT_DIR / "train.parquet", index=False)
val_df.to_parquet(OUT_DIR / "val.parquet", index=False)
test_df.to_parquet(OUT_DIR / "test.parquet", index=False)
joblib.dump(pipe, OUT_DIR / "model.joblib")
print("artifacts written to", OUT_DIR.resolve())